# Geumo Reservoir Dam-Break Simulation - Verification Suite
## For Sustainability Journal Submission

**Purpose**: Three verification tests to validate the GPU-accelerated SWE solver

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| V1 | 1D Dam-Break vs Ritter (1892) analytical solution | NSE > 0.90 |
| V2 | Grid convergence study (5 resolutions) | Order ~ 1.0 |
| V3 | Mass conservation (main simulation) | Error < 5% |

In [ ]:
# =============================================================================
# Cell 1: Install & Import
# =============================================================================
import subprocess, sys

for pkg in ['cupy-cuda12x', 'numba', 'numpy', 'matplotlib', 'h5py', 'tqdm']:
    try:
        name = pkg.split('-')[0].split('=')[0].lower()
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from numba import cuda
import math
import time as _time
import os
import json

VERI_DIR = os.path.join(os.getcwd(), 'verification_output')
os.makedirs(VERI_DIR, exist_ok=True)

device = cp.cuda.Device(0)
free_mem, total_mem = device.mem_info
props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props['name'].decode()

print(f'GPU: {gpu_name}')
print(f'Memory: {free_mem/1e9:.1f} / {total_mem/1e9:.1f} GB free')
print(f'Output: {VERI_DIR}')
print('[OK] Setup complete.')

In [ ]:
# =============================================================================
# Cell 2: Define CUDA Kernels (same as main simulation)
# =============================================================================

@cuda.jit
def swe_step(z, h, hu, hv, h_new, hu_new, hv_new,
             max_h, max_v, arrival,
             nx, ny, dx, dy, dt, g, manning_n, dry_thresh, current_time):
    j, i = cuda.grid(2)
    if i < 1 or i >= nx - 1 or j < 1 or j >= ny - 1:
        return

    h_c  = h[j, i];     h_l  = h[j, i-1];   h_r  = h[j, i+1]
    h_u  = h[j-1, i];   h_d  = h[j+1, i]
    hu_c = hu[j, i];    hu_l = hu[j, i-1];   hu_r = hu[j, i+1]
    hu_u = hu[j-1, i];  hu_d = hu[j+1, i]
    hv_c = hv[j, i];    hv_l = hv[j, i-1];   hv_r = hv[j, i+1]
    hv_u = hv[j-1, i];  hv_d = hv[j+1, i]
    z_c  = z[j, i];     z_l  = z[j, i-1];   z_r  = z[j, i+1]
    z_u  = z[j-1, i];   z_d  = z[j+1, i]

    h_avg  = 0.25 * (h_l + h_r + h_u + h_d)
    hu_avg = 0.25 * (hu_l + hu_r + hu_u + hu_d)
    hv_avg = 0.25 * (hv_l + hv_r + hv_u + hv_d)

    if h_c > dry_thresh:
        u_c = hu_c / h_c
        v_c = hv_c / h_c
    else:
        u_c = 0.0
        v_c = 0.0

    h_L = 0.5 * (h_l + h_c)
    if h_L > dry_thresh:
        u_L = 0.5 * (hu_l / max(h_l, dry_thresh) + u_c)
    else:
        u_L = 0.0
    fh_xL  = h_L * u_L
    fhu_xL = h_L * u_L * u_L + 0.5 * g * h_L * h_L

    h_R = 0.5 * (h_c + h_r)
    if h_R > dry_thresh:
        u_R = 0.5 * (u_c + hu_r / max(h_r, dry_thresh))
    else:
        u_R = 0.0
    fh_xR  = h_R * u_R
    fhu_xR = h_R * u_R * u_R + 0.5 * g * h_R * h_R

    h_U = 0.5 * (h_u + h_c)
    if h_U > dry_thresh:
        v_U = 0.5 * (hv_u / max(h_u, dry_thresh) + v_c)
    else:
        v_U = 0.0
    fh_yU  = h_U * v_U
    fhv_yU = h_U * v_U * v_U + 0.5 * g * h_U * h_U

    h_D = 0.5 * (h_c + h_d)
    if h_D > dry_thresh:
        v_D = 0.5 * (v_c + hv_d / max(h_d, dry_thresh))
    else:
        v_D = 0.0
    fh_yD  = h_D * v_D
    fhv_yD = h_D * v_D * v_D + 0.5 * g * h_D * h_D

    Sx = -g * h_c * (z_r - z_l) / (2.0 * dx)
    Sy = -g * h_c * (z_d - z_u) / (2.0 * dy)

    h_new[j, i]  = h_avg  - (dt / dx) * (fh_xR - fh_xL)   - (dt / dy) * (fh_yD - fh_yU)
    hu_new[j, i] = hu_avg - (dt / dx) * (fhu_xR - fhu_xL) + dt * Sx
    hv_new[j, i] = hv_avg - (dt / dy) * (fhv_yD - fhv_yU) + dt * Sy

    h_n = h_new[j, i]
    if h_n > dry_thresh:
        u_n = hu_new[j, i] / h_n
        v_n = hv_new[j, i] / h_n
        speed = math.sqrt(u_n * u_n + v_n * v_n)
        if speed > 1.0e-6:
            Sf = manning_n * manning_n * speed / (h_n ** (4.0 / 3.0))
            friction = 1.0 / (1.0 + dt * g * Sf / speed)
            hu_new[j, i] *= friction
            hv_new[j, i] *= friction
    else:
        h_new[j, i]  = 0.0
        hu_new[j, i] = 0.0
        hv_new[j, i] = 0.0

    if h_new[j, i] < 0.0:
        h_new[j, i]  = 0.0
        hu_new[j, i] = 0.0
        hv_new[j, i] = 0.0

    if h_new[j, i] > max_h[j, i]:
        max_h[j, i] = h_new[j, i]
    if h_n > dry_thresh:
        spd = math.sqrt((hu_new[j, i] / h_n)**2 + (hv_new[j, i] / h_n)**2)
        if spd > max_v[j, i]:
            max_v[j, i] = spd
    if h_new[j, i] > 0.01 and arrival[j, i] < 0.0:
        arrival[j, i] = current_time


@cuda.jit
def compute_max_wavespeed(h, hu, hv, result, nx, ny, g, dry_thresh):
    j, i = cuda.grid(2)
    if i >= nx or j >= ny:
        return
    hh = h[j, i]
    if hh > dry_thresh:
        u_abs = abs(hu[j, i] / hh)
        v_abs = abs(hv[j, i] / hh)
        c = math.sqrt(g * hh)
        result[j, i] = max(u_abs + c, v_abs + c)
    else:
        result[j, i] = 0.0

print('[OK] CUDA kernels compiled.')

In [ ]:
# =============================================================================
# Cell 3: Helper functions
# =============================================================================

def ritter_exact(x, t, h_L=10.0, g=9.81, x_dam=0.0):
    # Ritter (1892) exact solution for instantaneous dam-break
    # on flat frictionless bed. h_R = 0 (dry bed).
    c0 = np.sqrt(g * h_L)
    h_exact = np.zeros_like(x, dtype=np.float64)
    u_exact = np.zeros_like(x, dtype=np.float64)

    if t <= 0:
        h_exact[x < x_dam] = h_L
        return h_exact, u_exact

    xi = (x - x_dam) / t

    mask1 = xi <= -c0
    h_exact[mask1] = h_L
    u_exact[mask1] = 0.0

    mask2 = (~mask1) & (xi < 2.0 * c0)
    h_exact[mask2] = (1.0 / (9.0 * g)) * (2.0 * c0 - xi[mask2]) ** 2
    u_exact[mask2] = (2.0 / 3.0) * (c0 + xi[mask2])

    mask3 = xi >= 2.0 * c0
    h_exact[mask3] = 0.0
    u_exact[mask3] = 0.0

    return h_exact, u_exact


def run_1d_dambreak(v_nx, v_dx, h_L, target_time, manning=0.0,
                    cfl=0.40, g=9.81, dry_thresh=0.001):
    # Run 1D dam-break using 2D SWE kernel on thin strip
    v_ny = 5
    v_dam = v_nx // 2
    v_dx_f = np.float32(v_dx)
    v_dy_f = np.float32(v_dx)
    v_g = np.float32(g)
    v_n = np.float32(manning)
    v_dry = np.float32(dry_thresh)

    _z   = cp.zeros((v_ny, v_nx), dtype=cp.float32)
    _h   = cp.zeros((v_ny, v_nx), dtype=cp.float32)
    _hu  = cp.zeros((v_ny, v_nx), dtype=cp.float32)
    _hv  = cp.zeros((v_ny, v_nx), dtype=cp.float32)
    _hn  = cp.zeros_like(_h)
    _hun = cp.zeros_like(_hu)
    _hvn = cp.zeros_like(_hv)
    _mh  = cp.zeros_like(_h)
    _mv  = cp.zeros_like(_h)
    _ar  = cp.full_like(_h, -1.0)
    _ws  = cp.zeros_like(_h)

    _h[:, :v_dam] = cp.float32(h_L)

    threads = (16, 16)
    blocks = (
        (v_nx + threads[0] - 1) // threads[0],
        (v_ny + threads[1] - 1) // threads[1],
    )

    sim_t = 0.0
    step = 0
    t0 = _time.time()

    while sim_t < target_time:
        compute_max_wavespeed[blocks, threads](
            _h, _hu, _hv, _ws, v_nx, v_ny, v_g, v_dry)
        mws = float(cp.max(_ws))
        dt = cfl * float(v_dx) / max(mws, 1e-6)
        dt = min(dt, 0.5, target_time - sim_t)
        dt = max(dt, 1e-6)

        swe_step[blocks, threads](
            _z, _h, _hu, _hv, _hn, _hun, _hvn,
            _mh, _mv, _ar,
            v_nx, v_ny, v_dx_f, v_dy_f, cp.float32(dt),
            v_g, v_n, v_dry, cp.float32(sim_t))

        _hn[0,:]=_hn[1,:]; _hn[-1,:]=_hn[-2,:]
        _hn[:,0]=_hn[:,1]; _hn[:,-1]=_hn[:,-2]
        _hun[0,:]=_hun[1,:]; _hun[-1,:]=_hun[-2,:]
        _hun[:,0]=_hun[:,1]; _hun[:,-1]=_hun[:,-2]
        _hvn[0,:]=_hvn[1,:]; _hvn[-1,:]=_hvn[-2,:]
        _hvn[:,0]=_hvn[:,1]; _hvn[:,-1]=_hvn[:,-2]

        _h, _hn = _hn, _h
        _hu, _hun = _hun, _hu
        _hv, _hvn = _hvn, _hv
        sim_t += dt
        step += 1

    wall_t = _time.time() - t0
    mid = v_ny // 2
    h_row = cp.asnumpy(_h[mid, :])
    hu_row = cp.asnumpy(_hu[mid, :])
    x_coords = (np.arange(v_nx) + 0.5) * float(v_dx)

    del _z, _h, _hu, _hv, _hn, _hun, _hvn, _mh, _mv, _ar, _ws
    cp.cuda.Stream.null.synchronize()

    return x_coords, h_row, hu_row, sim_t, step, wall_t


def compute_metrics(h_num, h_exact, threshold=0.01):
    # Compute RMSE, MAE, NSE, Relative L2 error for wet cells
    wet = h_exact > threshold
    if np.sum(wet) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'nse': np.nan, 'rel_l2': np.nan}
    diff = h_num[wet] - h_exact[wet]
    rmse = np.sqrt(np.mean(diff**2))
    mae  = np.mean(np.abs(diff))
    ss_res = np.sum(diff**2)
    ss_tot = np.sum((h_exact[wet] - np.mean(h_exact[wet]))**2)
    nse = 1.0 - ss_res / max(ss_tot, 1e-10)
    rel_l2 = np.sqrt(ss_res) / np.sqrt(np.sum(h_exact[wet]**2))
    return {'rmse': rmse, 'mae': mae, 'nse': nse, 'rel_l2': rel_l2}

print('[OK] Helper functions defined.')

In [ ]:
# =============================================================================
# Cell 4: VERIFICATION 1 - 1D Dam-Break vs Ritter Analytical Solution
# =============================================================================
print('=' * 65)
print('  VERIFICATION 1: 1D Dam-Break vs Ritter (1892) Exact Solution')
print('=' * 65)
print()

V1_NX = 2000
V1_DX = 0.5
V1_HL = 10.0
V1_TIMES = [2.0, 5.0, 10.0, 20.0]

v1_results = {}
t_wall_total = 0.0

for t_target in V1_TIMES:
    x, h_num, hu_num, t_actual, steps, wt = run_1d_dambreak(
        V1_NX, V1_DX, V1_HL, t_target, manning=0.0)
    t_wall_total += wt

    x_dam = (V1_NX // 2) * V1_DX
    h_ex, u_ex = ritter_exact(x, t_actual, h_L=V1_HL, x_dam=x_dam)
    u_num = np.where(h_num > 0.001, hu_num / h_num, 0.0)

    met = compute_metrics(h_num, h_ex)
    met_u = compute_metrics(u_num, u_ex, threshold=0.01)

    v1_results[t_target] = {
        'x': x, 'h_num': h_num, 'h_exact': h_ex,
        'u_num': u_num, 'u_exact': u_ex,
        'metrics_h': met, 'metrics_u': met_u,
        'steps': steps, 'wall_time': wt
    }

    print(f'  t = {t_target:5.1f} s | RMSE_h = {met["rmse"]:.4f} m | '
          f'NSE_h = {met["nse"]:.4f} | steps = {steps:,} | {wt:.2f} s')

print(f'\n  Total wall-clock: {t_wall_total:.1f} s')

all_nse = [v1_results[t]['metrics_h']['nse'] for t in V1_TIMES]
min_nse = min(all_nse)
if min_nse >= 0.90:
    print(f'\n  >>> PASS: All NSE >= 0.90 (min = {min_nse:.4f})')
elif min_nse >= 0.80:
    print(f'\n  >>> MARGINAL: min NSE = {min_nse:.4f}')
else:
    print(f'\n  >>> FAIL: min NSE = {min_nse:.4f}')

In [ ]:
# =============================================================================
# Cell 5: VERIFICATION 1 - Visualization
# =============================================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for idx, t_val in enumerate(V1_TIMES):
    r = v1_results[t_val]

    ax = axes[0, idx]
    ax.plot(r['x'], r['h_exact'], 'k-', linewidth=2.5, label='Exact (Ritter)')
    ax.plot(r['x'], r['h_num'], 'r-', linewidth=1.2, alpha=0.85, label='Numerical')
    ax.set_title(f't = {t_val:.0f} s', fontsize=13, fontweight='bold')
    ax.set_ylim(-0.5, V1_HL * 1.3)
    if idx == 0:
        ax.set_ylabel('Depth h (m)')
        ax.legend(fontsize=9, loc='upper right')
    ax.set_xlabel('x (m)')
    ax.grid(True, alpha=0.3)

    met = r['metrics_h']
    ax.text(0.03, 0.03,
            f'RMSE={met["rmse"]:.3f}m\nNSE={met["nse"]:.3f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax = axes[1, idx]
    ax.plot(r['x'], r['u_exact'], 'k-', linewidth=2.5, label='Exact')
    ax.plot(r['x'], r['u_num'], 'b-', linewidth=1.2, alpha=0.85, label='Numerical')
    ax.set_ylim(-2, 22)
    if idx == 0:
        ax.set_ylabel('Velocity u (m/s)')
        ax.legend(fontsize=9, loc='upper right')
    ax.set_xlabel('x (m)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Verification 1: 1D Dam-Break vs Ritter (1892) Analytical Solution',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(VERI_DIR, 'V1_ritter_dambreak.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(VERI_DIR, 'V1_ritter_dambreak.pdf'), bbox_inches='tight')
plt.show()

print('\nSaved: V1_ritter_dambreak.png & .pdf')

print(f'\n{"="*75}')
print(f'{"Time (s)":>10}  {"RMSE_h (m)":>12}  {"MAE_h (m)":>12}  '
      f'{"NSE_h":>8}  {"Rel.L2_h":>10}')
print(f'{"-"*75}')
for t_val in V1_TIMES:
    r = v1_results[t_val]
    m = r['metrics_h']
    print(f'{t_val:>10.1f}  {m["rmse"]:>12.4f}  {m["mae"]:>12.4f}  '
          f'{m["nse"]:>8.4f}  {m["rel_l2"]:>10.4f}')
print(f'{"="*75}')

In [ ]:
# =============================================================================
# Cell 6: VERIFICATION 2 - Grid Convergence Study
# =============================================================================
print('=' * 65)
print('  VERIFICATION 2: Grid Convergence Study')
print('=' * 65)

CONV_RESOLUTIONS = [4.0, 2.0, 1.0, 0.5, 0.25]
CONV_TIME = 10.0
CONV_HL = 10.0

v2_results = {}

for res in CONV_RESOLUTIONS:
    cnx = int(1000.0 / res)
    print(f'  dx = {res:5.2f} m | cells = {cnx:>6,} ... ', end='', flush=True)

    x, h_num, hu_num, t_act, steps, wt = run_1d_dambreak(
        cnx, res, CONV_HL, CONV_TIME, manning=0.0)

    x_dam = (cnx // 2) * res
    h_ex, u_ex = ritter_exact(x, t_act, h_L=CONV_HL, x_dam=x_dam)
    met = compute_metrics(h_num, h_ex)

    v2_results[res] = {
        'x': x, 'h_num': h_num, 'h_exact': h_ex,
        'metrics': met, 'steps': steps, 'wall_time': wt, 'ncells': cnx
    }

    print(f'RMSE = {met["rmse"]:.4f} m | NSE = {met["nse"]:.4f} | '
          f'{wt:.2f} s | {steps:,} steps')

dx_arr = np.array(CONV_RESOLUTIONS)
rmse_arr = np.array([v2_results[r]['metrics']['rmse'] for r in CONV_RESOLUTIONS])
orders = []
for i in range(len(CONV_RESOLUTIONS) - 1):
    if rmse_arr[i+1] > 0 and rmse_arr[i] > 0:
        p = np.log(rmse_arr[i] / rmse_arr[i+1]) / np.log(dx_arr[i] / dx_arr[i+1])
        orders.append(p)
    else:
        orders.append(np.nan)
avg_order = np.nanmean(orders)

print(f'\n  Average convergence order: {avg_order:.2f}')
if 0.7 <= avg_order <= 1.3:
    print('  >>> PASS: Consistent with 1st-order Lax-Friedrichs scheme')
else:
    print(f'  >>> CHECK: Expected ~1.0, got {avg_order:.2f}')

In [ ]:
# =============================================================================
# Cell 7: VERIFICATION 2 - Visualization
# =============================================================================

fig = plt.figure(figsize=(18, 7))
gs = gridspec.GridSpec(1, 3, width_ratios=[1.2, 1, 1])

ax = fig.add_subplot(gs[0])
colors = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']
for idx, res in enumerate(CONV_RESOLUTIONS):
    r = v2_results[res]
    ax.plot(r['x'], r['h_num'], color=colors[idx % len(colors)],
            linewidth=1.0, alpha=0.85,
            label=f'dx={res}m (RMSE={r["metrics"]["rmse"]:.3f})')
r_finest = v2_results[CONV_RESOLUTIONS[-1]]
ax.plot(r_finest['x'], r_finest['h_exact'], 'k-', linewidth=2.5,
        label='Exact (Ritter)', zorder=10)
ax.set_title(f'(a) Depth Profiles at t = {CONV_TIME:.0f} s', fontweight='bold', fontsize=12)
ax.set_xlabel('x (m)'); ax.set_ylabel('Water Depth (m)')
ax.set_ylim(-0.5, 13); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1])
ax.loglog(dx_arr, rmse_arr, 'bo-', markersize=10, linewidth=2.5, label='RMSE', zorder=5)
dx_ref = np.array([dx_arr[0], dx_arr[-1]])
rmse_ref1 = rmse_arr[0] * (dx_ref / dx_arr[0]) ** 1.0
ax.loglog(dx_ref, rmse_ref1, 'k--', linewidth=1.5, alpha=0.5, label='1st order slope')
for i, res in enumerate(CONV_RESOLUTIONS):
    ax.annotate(f'{res}m', (dx_arr[i], rmse_arr[i]),
                textcoords='offset points', xytext=(8, 5), fontsize=9)
ax.set_title(f'(b) Convergence Rate (avg. order = {avg_order:.2f})',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Grid spacing dx (m)'); ax.set_ylabel('RMSE (m)')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, which='both')

ax = fig.add_subplot(gs[2])
nse_arr = [v2_results[r]['metrics']['nse'] for r in CONV_RESOLUTIONS]
ax.semilogx(dx_arr, nse_arr, 'gs-', markersize=10, linewidth=2.5)
ax.axhline(0.90, color='red', linestyle='--', alpha=0.5, label='NSE = 0.90')
for i, res in enumerate(CONV_RESOLUTIONS):
    ax.annotate(f'{nse_arr[i]:.3f}', (dx_arr[i], nse_arr[i]),
                textcoords='offset points', xytext=(8, -12), fontsize=9)
ax.set_title('(c) NSE vs Grid Spacing', fontweight='bold', fontsize=12)
ax.set_xlabel('Grid spacing dx (m)'); ax.set_ylabel('NSE')
ax.set_ylim(0.5, 1.05); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('Verification 2: Grid Convergence Study', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(VERI_DIR, 'V2_grid_convergence.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(VERI_DIR, 'V2_grid_convergence.pdf'), bbox_inches='tight')
plt.show()

print('Saved: V2_grid_convergence.png & .pdf')

print(f'\n{"="*80}')
print(f'{"dx (m)":>8}  {"Cells":>8}  {"RMSE (m)":>10}  {"NSE":>8}  '
      f'{"Rel.L2":>8}  {"Order":>8}')
print(f'{"-"*80}')
for i, res in enumerate(CONV_RESOLUTIONS):
    r = v2_results[res]
    m = r['metrics']
    o_str = f'{orders[i-1]:.2f}' if i > 0 else '  --'
    print(f'{res:>8.2f}  {r["ncells"]:>8,}  {m["rmse"]:>10.4f}  '
          f'{m["nse"]:>8.4f}  {m["rel_l2"]:>8.4f}  {o_str:>8}')
print(f'{"="*80}')
print(f'  Average convergence order: {avg_order:.2f}')

In [ ]:
# =============================================================================
# Cell 8: VERIFICATION 3 - Mass Conservation from Main Simulation
# Requires: output/geumo_dambreak_results.h5 from main notebook
# =============================================================================
import h5py

print('=' * 65)
print('  VERIFICATION 3: Mass (Volume) Conservation')
print('=' * 65)

hdf_candidates = [
    os.path.join(os.getcwd(), 'output', 'geumo_dambreak_results.h5'),
    os.path.join(os.getcwd(), 'geumo_dambreak_results.h5'),
    'output/geumo_dambreak_results.h5',
]

hdf_path = None
for cand in hdf_candidates:
    if os.path.exists(cand):
        hdf_path = cand
        break

if hdf_path is None:
    print('\n  [WARNING] HDF5 result file not found!')
    print('  Run the main simulation notebook first, then re-run this cell.')
    V3_AVAILABLE = False
else:
    V3_AVAILABLE = True
    print(f'  Found: {hdf_path}')

    with h5py.File(hdf_path, 'r') as f:
        ts_time = f['time_series/time_s'][:]
        ts_vol  = f['time_series/total_volume_m3'][:]
        ts_maxd = f['time_series/max_depth_m'][:]
        ts_wet  = f['time_series/wet_area_km2'][:]
        ts_vel  = f['time_series/max_velocity_ms'][:]

    t_min = ts_time / 60.0
    init_v = ts_vol[0]
    rel_error_pct = (ts_vol - init_v) / init_v * 100.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

    ax = axes[0]
    ax.plot(t_min, ts_vol / 1e6, 'b-', linewidth=2)
    ax.axhline(init_v / 1e6, color='red', linestyle='--', linewidth=1.5,
               label=f'Initial = {init_v/1e6:.3f} x 10^6 m3')
    ax.set_title('(a) Total Water Volume', fontweight='bold', fontsize=12)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('Volume (x 10^6 m3)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(t_min, rel_error_pct, 'r-', linewidth=2)
    ax.fill_between(t_min, -5, 5, color='green', alpha=0.1, label='pm 5% band')
    ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
    ax.set_title('(b) Volume Conservation Error', fontweight='bold', fontsize=12)
    ax.set_xlabel('Time (min)'); ax.set_ylabel('Relative Error (%)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax2 = ax.twinx()
    l1, = ax.plot(t_min, ts_maxd, 'b-', linewidth=2, label='Max Depth')
    l2, = ax2.plot(t_min, ts_wet, 'g-', linewidth=2, label='Wet Area')
    ax.set_title('(c) Flood Evolution', fontweight='bold', fontsize=12)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Max Depth (m)', color='blue')
    ax2.set_ylabel('Wet Area (km2)', color='green')
    ax.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.suptitle('Verification 3: Mass Conservation', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(VERI_DIR, 'V3_mass_conservation.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(VERI_DIR, 'V3_mass_conservation.pdf'), bbox_inches='tight')
    plt.show()

    max_err = np.max(np.abs(rel_error_pct))
    final_err = rel_error_pct[-1]
    print(f'\n  Initial volume  : {init_v:>14,.0f} m3')
    print(f'  Final volume    : {ts_vol[-1]:>14,.0f} m3')
    print(f'  Max |error|     : {max_err:>10.2f} %')
    print(f'  Final error     : {final_err:>10.2f} %')

    if max_err < 5.0:
        print('\n  >>> PASS: Volume conservation error < 5%')
    elif max_err < 10.0:
        print('\n  >>> MARGINAL: Volume conservation error 5-10%')
    else:
        print('\n  >>> FAIL: Volume conservation error > 10%')

    print('\nSaved: V3_mass_conservation.png & .pdf')

In [ ]:
# =============================================================================
# Cell 9: Export all verification results
# =============================================================================

export_data = {
    'gpu': gpu_name,
    'verification_1': {},
    'verification_2': {
        'average_order': round(float(avg_order), 4)
    }
}

for t_val in V1_TIMES:
    m = v1_results[t_val]['metrics_h']
    key = f't_{t_val:.0f}s'
    export_data['verification_1'][key] = {
        'rmse_m': round(float(m['rmse']), 6),
        'mae_m':  round(float(m['mae']), 6),
        'nse':    round(float(m['nse']), 6),
        'rel_l2': round(float(m['rel_l2']), 6),
    }

for res in CONV_RESOLUTIONS:
    m = v2_results[res]['metrics']
    key = f'dx_{res}m'
    export_data['verification_2'][key] = {
        'rmse_m': round(float(m['rmse']), 6),
        'nse':    round(float(m['nse']), 6),
    }

has_v3 = 'V3_AVAILABLE' in dir() and V3_AVAILABLE
if has_v3:
    export_data['verification_3'] = {
        'initial_volume_m3': float(init_v),
        'final_volume_m3': float(ts_vol[-1]),
        'max_error_pct': round(float(np.max(np.abs(rel_error_pct))), 4),
    }

json_path = os.path.join(VERI_DIR, 'verification_results.json')
with open(json_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print('\n' + '=' * 60)
print('  VERIFICATION OUTPUT FILES')
print('=' * 60)
total_size = 0
for fname in sorted(os.listdir(VERI_DIR)):
    fpath = os.path.join(VERI_DIR, fname)
    fsize = os.path.getsize(fpath)
    total_size += fsize
    if fsize > 1e6:
        s = f'{fsize/1e6:.1f} MB'
    else:
        s = f'{fsize/1e3:.0f} KB'
    print(f'  {fname:45s} {s:>10}')
print(f'  {"Total:":45s} {total_size/1e6:.1f} MB')
print('=' * 60)
print('\n  ALL VERIFICATION COMPLETE.')

---
## Quick Reference

```bash
# Upload to supercomputer
scp verification_suite.ipynb user@supercomputer:/home/work/gumi/

# Download results
scp -r user@supercomputer:/home/work/gumi/verification_output/ ./
```

| Component | Time | Cost |
|-----------|------|------|
| V1 + V2 | ~5 min | ~125 KRW |
| Main sim | ~20 min | ~500 KRW |
| V3 | ~1 min | ~25 KRW |
| **Total** | **~26 min** | **~650 KRW** |